# 05 — Handling message blocks

When working with Claude's tool functionality, you'll encounter a new type of response structure that's different from the simple text responses you've seen before. Instead of just getting back a single text block, Claude can now return multi-block messages that contain both text and tool usage information.

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

#  Imports
from src.utils import get_client, get_model

client = get_client()
model = get_model()

In [2]:
DATETIME_FORMAT = "%Y-%m-%d %H:%M:%S"

In [3]:
from datetime import datetime


def get_current_datetime(date_format=DATETIME_FORMAT):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

In [4]:
from anthropic.types import ToolParam


get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time formatted according to a specified format string. Use this tool when the user asks about the current date, time, or both. Do not use this tool if the user is asking about a historical or future date — it only returns the current moment. The format string follows Python's strftime directives (e.g., '%Y-%m-%d' for ISO date, '%H:%M:%S' for time, '%Y-%m-%d %H:%M:%S' for full datetime). An empty format string will raise an error and must be avoided.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime-compatible format string that controls the structure of the returned datetime string. For example, '%Y-%m-%d' returns '2025-04-01', '%H:%M:%S' returns '14:30:00', and '%Y-%m-%d %H:%M:%S' returns '2025-04-01 14:30:00'. Must not be an empty string. Defaults to the system DATETIME_FORMAT constant if omitted.",
        "default": DATETIME_FORMAT
      }
    },
    "required": []
  },
  "input_examples": [
    { "date_format": "%Y-%m-%d" },
    { "date_format": "%H:%M:%S" },
    { "date_format": "%Y-%m-%d %H:%M:%S" },
    { "date_format": "%A, %B %d %Y" }
  ]
})

### Making Tool-Enabled API Calls
To enable Claude to use tools, you need to include a tools parameter in your API call. Here's how to structure the request:

In [5]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

### Understanding Multi-Block Messages
When Claude decides to use a tool, it returns an assistant message with multiple blocks in the content list. This is a significant change from the simple text-only responses you've worked with before.

A multi-block message typically contains:

- **Text Block** - Human-readable text explaining what Claude is doing (like "I can help you find out the current time. Let me find that information for you")
- **ToolUse Block** - Instructions for your code about which tool to call and what parameters to use

The ToolUse block includes:

- An ID for tracking the tool call
- The name of the function to call (like "get_current_datetime")
- Input parameters formatted as a dictionary
- The type designation "tool_use"

In [6]:
response

Message(id='msg_01JYcWQmvX7gnsoo1aQZ1x9e', container=None, content=[ToolUseBlock(id='toolu_01HcMQPtx5rNXLaYkqE1DsQH', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=890, output_tokens=62, server_tool_use=None, service_tier='standard'), stop_details=None)

### Managing Conversation History with Multi-Block Messages
Remember that Claude doesn't store conversation history - you need to manage it manually. When working with tool responses, you must preserve the entire content structure, including all blocks.

In [7]:
messages.append({
    "role": "assistant",
    "content": response.content
})

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01HcMQPtx5rNXLaYkqE1DsQH', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

The tool usage process follows this pattern:

- Send user message with tool schema to Claude
- Receive assistant message with text block and tool use block
- Extract tool information and execute the actual function
- Send tool result back to Claude along with complete conversation history
- Receive final response from Claude
- Each step requires careful handling of the message structure to ensure Claude has the full context it needs to provide accurate responses.